# 6. Decision Tree
**Use case:** Insurance — Homeowners underwriting decision: Accept / Refer / Decline

A homeowners insurer receives thousands of new-business applications a week and every one used to sit in an underwriter's queue. Most are clean risks that could be accepted straight away, a smaller group needs a human to look at roof photos or claim history, and a few are clearly outside appetite. The insurer has historic applications with roof age, prior claims, wildfire zone, construction type, distance to fire station, a credit-based insurance score and the final underwriter decision. Success means auto-accepting the clean risks, sending true edge cases to underwriters, almost never auto-accepting a risk that should be declined, and being able to print the exact rule behind every decision for regulators.

> 💡 It is like the game of 20 Questions played by a seasoned underwriter: 'Is the roof older than 20 years? Any claims in the last 5 years? Is the house in a wildfire zone?' Each answer sends you down a different branch until you land on Accept, Refer or Decline.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Hand-build the underwriting rule book from 12 applications
Compute Gini impurity by hand for one candidate split, then let scikit-learn grow a 3-level tree and print its rules.

| 5W + H | |
|---|---|
| **Who** | A new underwriting analyst who wants to see how a tree turns past decisions into rules. |
| **What** | Twelve typed-in homeowners applications with roof age, prior claims, wildfire zone and home value, each labelled Accept, Refer or Decline. |
| **When** | During a lunch-and-learn before the insurer pilots automated triage. |
| **Where** | A laptop Python session; no database, the data is inside the script. |
| **Why** | To build intuition that a tree is just a stack of the purest yes/no questions. |
| **How** | Write a tiny Gini function, test the split roof_age_yrs <= 13, then fit DecisionTreeClassifier(max_depth=3) and print export_text. |

**🧠 ML concepts:** Gini impurity, Choosing the best split, Tree depth as a complexity limit, Readable rules with export_text, Predicting by walking the tree  
**🐍 Python concepts:** Lists of lists as a table, List comprehensions with if filters, zip to walk two lists together, set() to get unique values, Defining a function with def, f-strings with format specs

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

# 🐍 PY: a list of lists = a tiny table typed by hand; each inner list is one homeowners application
# columns: roof_age_yrs, prior_claims_5yr, wildfire_zone (0/1), home_value_k
applications = [
    [5, 0, 0, 250], [3, 0, 0, 410], [8, 1, 0, 300], [12, 0, 0, 180],
    [18, 1, 0, 220], [22, 2, 0, 275], [15, 0, 1, 350], [6, 0, 1, 520],
    [25, 3, 1, 190], [28, 2, 0, 160], [20, 3, 1, 240], [10, 1, 1, 330],
]
# 🐍 PY: a matching list of answers (the label), one per application, same order
decisions = ["Accept", "Accept", "Accept", "Accept",
             "Refer", "Refer", "Refer", "Refer",
             "Decline", "Decline", "Decline", "Refer"]
features = ["roof_age_yrs", "prior_claims_5yr", "wildfire_zone", "home_value_k"]

# 🧠 ML: Gini impurity = chance two random rows in a group disagree; 0 means the group is "pure"
def gini(labels):
    # 🐍 PY: set() removes duplicates so we loop once per distinct decision
    shares = [labels.count(c) / len(labels) for c in set(labels)]
    return 1 - sum(p * p for p in shares)

print(f"Gini of all 12 applications: {gini(decisions):.3f}")

# 🧠 ML: try one candidate question by hand: "roof_age_yrs <= 13?" and measure how pure each side gets
# 🐍 PY: zip pairs each application with its decision so we can filter both together
left = [d for row, d in zip(applications, decisions) if row[0] <= 13]
right = [d for row, d in zip(applications, decisions) if row[0] > 13]
print(f"roof <= 13 -> {left}  gini={gini(left):.3f}")
print(f"roof  > 13 -> {right} gini={gini(right):.3f}")

# 🧠 ML: the tree repeats this search over every column and threshold, keeping the purest split
# 🧠 ML: max_depth=3 keeps the rule book short enough for an underwriter to read
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(applications, decisions)

# 🧠 ML: export_text prints the learned if-else rules - this is the "explainable" part
print("\nLearned underwriting rules:")
print(export_text(tree, feature_names=features))

# 🐍 PY: predict expects a list of rows, so wrap a single new application in an outer list
new_app = [[14, 2, 1, 280]]
print("New application", new_app[0], "->", tree.predict(new_app)[0])

**Try this next:**
- Change the hand split from roof_age_yrs <= 13 to prior_claims_5yr <= 1 and compare the two Gini values.
- Set max_depth=1 and see which single question the tree considers most important.
- Add two more Decline rows with high claims and see how the printed rules change.

## MEDIUM: Learn underwriting rules from 3,000 applications
Train and test a depth-limited tree on realistic synthetic homeowners data and read its rules, importances and errors.

| 5W + H | |
|---|---|
| **Who** | The underwriting analytics team preparing a triage pilot. |
| **What** | 3,000 synthetic applications with six risk columns and an Accept/Refer/Decline label driven by a hidden guideline. |
| **When** | Before go-live, to check whether a tree can reproduce underwriter decisions on unseen applications. |
| **Where** | A Jupyter or VS Code notebook against an extract from the policy admin system. |
| **Why** | To measure accuracy on held-out data and see which risk factors drive the decision. |
| **How** | Stratified train/test split, DecisionTreeClassifier(max_depth=4, min_samples_leaf=30), confusion matrix, classification report, feature importance and the top of the rule book. |

**🧠 ML concepts:** Train/test split with stratify, Overfitting control with max_depth and min_samples_leaf, Confusion matrix and per-class recall, Impurity-based feature importance, Business-focused error check  
**🐍 Python concepts:** pandas DataFrame from a dict, np.select for multi-way labels, DataFrame.drop and column selection, Series.sort_values and .items(), String repetition for ASCII bars

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🧠 ML: fixed seed = same synthetic book of business every run, so results are reproducible
np.random.seed(42)
n = 3000

# 🐍 PY: a dict of column-name -> numpy array becomes a pandas DataFrame in one line
df = pd.DataFrame({
    "roof_age_yrs": np.random.randint(0, 35, n),
    "prior_claims_5yr": np.random.poisson(0.6, n),
    "wildfire_zone": np.random.binomial(1, 0.2, n),
    "dist_fire_station_mi": np.round(np.random.gamma(2, 2.5, n), 1),
    "credit_insurance_score": np.random.randint(500, 900, n),
    "home_value_k": np.random.randint(120, 900, n),
})

# 🧠 ML: hidden "true" underwriting risk points - the tree must rediscover this logic from data
risk = (
    (df.roof_age_yrs > 20) * 2 + df.prior_claims_5yr * 1.5 + df.wildfire_zone * 2
    + (df.dist_fire_station_mi > 10) * 1 + (df.credit_insurance_score < 620) * 1.5
    + np.random.normal(0, 0.8, n)
)
# 🐍 PY: np.select picks a label per row from the first condition that is True
df["decision"] = np.select([risk < 2.5, risk < 5], ["Accept", "Refer"], default="Decline")
print("Decision mix:", df.decision.value_counts().to_dict())

# 🐍 PY: drop() returns a copy without the label column; X = inputs, y = answer
X = df.drop(columns="decision")
y = df["decision"]
# 🧠 ML: stratify keeps the Accept/Refer/Decline mix the same in train and test
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# 🧠 ML: min_samples_leaf=30 stops the tree making rules from a handful of policies (overfitting)
tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=30, random_state=42)
tree.fit(X_tr, y_tr)
pred = tree.predict(X_te)

print(f"Train accuracy: {tree.score(X_tr, y_tr):.3f}")
print(f"Test accuracy : {accuracy_score(y_te, pred):.3f}\n")
labels = ["Accept", "Refer", "Decline"]
# 🧠 ML: confusion matrix rows = true decision, columns = model decision
cm = pd.DataFrame(confusion_matrix(y_te, pred, labels=labels), index=labels, columns=labels)
print("Confusion matrix (rows=true):\n", cm, "\n")
print(classification_report(y_te, pred, digits=2))

# 🧠 ML: feature_importances_ = how much each column reduced impurity across all splits
imp = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature importance:")
# 🐍 PY: .items() yields (index, value) pairs; '#' * k builds a simple ASCII bar
for name, v in imp.items():
    print(f"  {name:24s} {v:.3f} {'#' * int(v * 40)}")

# 🧠 ML: only the top 2 levels (depth 0-1) printed so an underwriter can read the core rules
rules = export_text(tree, feature_names=list(X.columns), max_depth=1)
print("\nTop of the rule book:\n" + rules)

# Business read-out
dec_as_acc = ((y_te == "Decline") & (pred == "Accept")).sum()
print(f"Business check: {dec_as_acc} true-Decline risks auto-accepted out of {len(y_te)} test apps")

**Try this next:**
- Remove min_samples_leaf and set max_depth=None; compare train vs test accuracy to see overfitting.
- Change criterion to 'entropy' and check whether the top split changes.
- Increase the wildfire weight in the hidden risk formula and watch its importance rise.

## HARD: Production underwriting tree with tuning, cost and reason codes
Build a tuned, class-balanced tree pipeline, compare it to a manual rule on business cost, explain decisions and save the model.

| 5W + H | |
|---|---|
| **Who** | A senior data scientist and the underwriting model-risk reviewer. |
| **What** | A Pipeline (one-hot encoding plus DecisionTreeClassifier) tuned by grid search, scored on macro-F1 and a dollar cost matrix, with permutation importance and per-applicant decision paths. |
| **When** | At model approval time before the tree is deployed behind the quoting API. |
| **Where** | The insurer's model development environment; the saved joblib file is promoted to the scoring service. |
| **Why** | Regulators and state DOIs expect transparent, documented underwriting logic, and the business needs fewer costly auto-accepts of bad risks. |
| **How** | GridSearchCV over depth, leaf size and pruning with 5-fold stratified CV, class_weight='balanced', cost matrix vs a rule of thumb, permutation importance, decision_path reason codes and joblib save. |

**🧠 ML concepts:** Pipelines with ColumnTransformer, Grid search with stratified cross-validation, Class imbalance via class_weight, Cost-sensitive evaluation, Permutation importance, Per-decision explanations with decision_path  
**🐍 Python concepts:** Functions with default arguments and docstrings, Module-level constants, numpy matrix element-wise multiply, DataFrame.iloc double brackets to keep 2-D, Tuple unpacking in a for loop, os.path and tempfile for safe saving

In [ ]:
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
SEED = 42
LABELS = ["Accept", "Refer", "Decline"]
NUM = ["roof_age_yrs", "prior_claims_5yr", "dist_fire_station_mi", "credit_insurance_score",
       "home_value_k", "year_built"]
CAT = ["construction", "wildfire_zone"]


# 🐍 PY: default arguments (n=4000) let callers override the size without changing the function
def make_book(n=4000, seed=SEED):
    """Synthetic homeowners applications with a hidden underwriting guideline."""
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "roof_age_yrs": rng.integers(0, 35, n),
        "prior_claims_5yr": rng.poisson(0.6, n),
        "dist_fire_station_mi": np.round(rng.gamma(2, 2.5, n), 1),
        "credit_insurance_score": rng.integers(500, 900, n),
        "home_value_k": rng.integers(120, 900, n),
        "year_built": rng.integers(1940, 2024, n),
        "construction": rng.choice(["frame", "masonry", "superior"], n, p=[0.6, 0.3, 0.1]),
        "wildfire_zone": rng.choice(["low", "moderate", "high"], n, p=[0.65, 0.25, 0.10]),
    })
    # 🧠 ML: hidden guideline = the 'truth' underwriters followed; the tree must rediscover it
    risk = ((df.roof_age_yrs > 20) * 2 + df.prior_claims_5yr * 1.5
            + df.wildfire_zone.map({"low": 0, "moderate": 1, "high": 2.5})
            + (df.construction == "frame") * 0.7 + (df.dist_fire_station_mi > 10)
            + (df.credit_insurance_score < 620) * 1.5 + rng.normal(0, 0.7, n))
    # 🐍 PY: np.select returns the label of the first True condition, else the default
    df["decision"] = np.select([risk < 2.5, risk < 5], ["Accept", "Refer"], default="Decline")
    return df


def build_pipeline():
    # 🧠 ML: trees need no scaling, but text categories must become 0/1 columns first
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
        ("num", "passthrough", NUM),
    ], verbose_feature_names_out=False)
    # 🧠 ML: class_weight='balanced' makes rare Declines count more so the tree does not ignore them
    tree = DecisionTreeClassifier(class_weight="balanced", random_state=SEED)
    return Pipeline([("pre", pre), ("tree", tree)])


def business_cost(y_true, y_pred):
    # 🧠 ML: cost matrix rows=true, cols=decision; auto-accepting a Decline risk is the worst error
    cost = np.array([[0, 40, 300],      # true Accept: refer wastes underwriter time, decline loses premium
                     [150, 0, 120],     # true Refer
                     [2500, 200, 0]])   # true Decline: accepting it = expected claim leakage
    # 🐍 PY: cm * cost multiplies element by element; .sum() totals dollars across all cells
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    return int((cm * cost).sum())


df = make_book()
X, y = df.drop(columns="decision"), df["decision"]
# 🧠 ML: stratify keeps the Accept/Refer/Decline mix identical in train and test
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print("Class mix (train):", y_tr.value_counts().to_dict())

# 🧠 ML: tune depth and leaf size with 5-fold CV; macro-F1 treats all three decisions equally
grid = {"tree__max_depth": [3, 4, 5, 6, 8],
        "tree__min_samples_leaf": [10, 25, 50],
        "tree__ccp_alpha": [0.0, 0.001]}
# 🧠 ML: 5 folds = 5 different train/validate splits; the average score is more trustworthy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
search = GridSearchCV(build_pipeline(), grid, scoring="f1_macro", cv=cv, n_jobs=1)
search.fit(X_tr, y_tr)
best = search.best_estimator_
print("Best params:", search.best_params_, f"CV macro-F1={search.best_score_:.3f}")

# 🧠 ML: the test set was never seen during tuning - this is the honest performance estimate
pred = best.predict(X_te)
print(f"Test macro-F1: {f1_score(y_te, pred, average='macro'):.3f}")
print("Confusion (rows=true A/R/D):", confusion_matrix(y_te, pred, labels=LABELS).tolist())

# 🧠 ML: baseline = today's manual rule 'refer everything with any prior claim, else accept'
baseline = np.where(X_te.prior_claims_5yr > 0, "Refer", "Accept")
print(f"Business cost  model=${business_cost(y_te, pred):,}  vs  rule-of-thumb=${business_cost(y_te, baseline):,}")

# 🧠 ML: permutation importance = drop in score when one column is shuffled (model-agnostic check)
pi = permutation_importance(best, X_te, y_te, scoring="f1_macro", n_repeats=5, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=X_te.columns).sort_values(ascending=False)
print("\nPermutation importance (macro-F1 drop):")
for col, v in imp.head(6).items():
    print(f"  {col:24s} {v:+.3f} {'#' * max(0, int(v * 60))}")

# 🧠 ML: a shallow 'reason code' view: the printed tree IS the model, auditable line by line
# 🐍 PY: named_steps is a dict-like view of the pipeline stages by their names
names = list(best.named_steps["pre"].get_feature_names_out())
rules = export_text(best.named_steps["tree"], feature_names=names, max_depth=2)
print("\nTop rules (depth<=2):")
print("\n".join(rules.splitlines()[:12]))

# 🧠 ML: decision_path gives the exact rules one applicant passed through - an adverse-action reason
app = X_te.iloc[[0]]
t = best.named_steps["tree"]
Xa = best.named_steps["pre"].transform(app)
# 🐍 PY: .indices lists node ids on the path; the last one is the leaf, so we skip it below
node_ids = t.decision_path(Xa).indices
print(f"\nApplicant 0 -> {best.predict(app)[0]}; path:")
for nid in node_ids[:-1]:
    f, thr = t.tree_.feature[nid], t.tree_.threshold[nid]
    side = "<=" if Xa[0, f] <= thr else ">"
    print(f"  {names[f]} = {Xa[0, f]:.1f} {side} {thr:.1f}")

# Regulatory note: NAIC model-governance bulletins + state rules on credit-based insurance scores
# require explainable decisions, documented variables, and periodic unfair-discrimination testing.
print("\nModel-risk note: credit_insurance_score use must follow state law; log rules + version.")
# 🧠 ML: save the WHOLE pipeline (encoder + tree) so scoring uses identical preprocessing
out = os.path.join(tempfile.mkdtemp(), "ho_underwriting_tree.joblib")
joblib.dump(best, out)
print("Saved model ->", os.path.basename(out), f"({os.path.getsize(out)} bytes)")

**Try this next:**
- Raise the cost of a Decline-accepted error to 10,000 and add a post-rule that sends any Accept with prior_claims_5yr >= 2 to Refer; compare cost.
- Drop credit_insurance_score (some states restrict it) and measure how much macro-F1 you lose.
- Add max_leaf_nodes to the grid and find the smallest tree within 0.01 F1 of the best.